# EDA (Exploratory Data Analysis) — Estrategia E2 Moderate (LSTM, 20 días)

Análisis exploratorio de datos para la estrategia moderada de medio plazo.

**Tickers**: 5 argentinos (.BA) + 5 estadounidenses  
**Horizonte**: 20 días  
**Modelo**: LSTM  
**Features**: 16 indicadores técnicos  
**Datos**: OHLCV diario (~10 años)

## 0. Setup

In [ ]:
import sys
from pathlib import Path

# Agregar raíz del proyecto al path para poder importar módulos de src/
# Como este notebook vive en notebooks/eda/, subimos 2 niveles (../../)
ROOT = Path("../..").resolve()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

# --- Librerías de datos y visualización ---
import numpy as np       # Operaciones numéricas (log, arrays, etc.)
import pandas as pd      # DataFrames para datos tabulares
import matplotlib.pyplot as plt  # Gráficos base
import seaborn as sns    # Gráficos estadísticos (heatmaps, boxplots, etc.)
from scipy import stats  # Tests estadísticos (Jarque-Bera, Q-Q, Pearson, etc.)
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf  # Autocorrelación

# --- Módulos del proyecto ---
from src.utils import project_root, load_yaml              # Utilidades: ruta raíz, cargar YAML
from src.e2.build_features import compute_e2_features, make_target_e2  # Features técnicos y target
from src.e2.train_pipeline import load_ohlcv_csv           # Carga de CSV con parsing de timestamps
from src.data.clean_daily import diagnose_data_quality      # Diagnóstico de calidad de datos

# --- Estilo global de gráficos ---
sns.set_theme(style="whitegrid", palette="muted")  # Fondo con grilla suave, colores apagados
plt.rcParams["figure.figsize"] = (14, 6)           # Tamaño por defecto de figuras
plt.rcParams["figure.dpi"] = 100                   # Resolución de pantalla

print(f"Raíz del proyecto: {ROOT}")

In [ ]:
# --- Cargar configuración central del proyecto ---
# base.yaml contiene todos los parámetros: tickers, horizonte, lookback, modelo, etc.
cfg = load_yaml(project_root() / "src" / "config" / "base.yaml")

# Extraer la lista de 10 tickers asignados a la estrategia E2
E2_TICKERS = cfg["universe"]["tickers_by_strategy"]["e2_moderate"]

# Extraer parámetros de la estrategia E2
E2_CFG = cfg["strategies"]["e2_moderate"]
HORIZON = E2_CFG["horizon_days"]   # 20 días: cuánto hacia el futuro predecimos
LOOKBACK = E2_CFG["lookback_days"] # 60 días: cuántos días de historia ve el modelo como input

# --- Clasificar tickers por mercado ---
# Los tickers argentinos terminan en ".BA" (Buenos Aires)
AR_TICKERS = [t for t in E2_TICKERS if t.endswith(".BA")]
US_TICKERS = [t for t in E2_TICKERS if not t.endswith(".BA")]

print(f"Tickers E2: {E2_TICKERS}")
print(f"  AR ({len(AR_TICKERS)}): {AR_TICKERS}")
print(f"  US ({len(US_TICKERS)}): {US_TICKERS}")
print(f"Horizonte: {HORIZON} días | Lookback: {LOOKBACK} días")

In [ ]:
# --- Cargar datos OHLCV limpios (post-cleaning) para cada ticker ---
# Los CSV limpios están en data/clean/ con formato: timestamp, open, high, low, close, volume
# load_ohlcv_csv() parsea timestamps a datetime UTC y valida que existan las columnas requeridas.
# El resultado es un DataFrame indexado por timestamp (DatetimeIndex).
clean_dir = project_root() / "data" / "clean"
data = {}  # Diccionario {ticker: DataFrame_OHLCV}

for ticker in E2_TICKERS:
    path = clean_dir / f"{ticker}_daily.csv"
    if path.exists():
        data[ticker] = load_ohlcv_csv(path)
        print(f"  {ticker}: {len(data[ticker]):,} días "
              f"| {data[ticker].index.min().date()} → {data[ticker].index.max().date()}")
    else:
        print(f"  ⚠️ {ticker}: archivo no encontrado en {path}")

print(f"\nTickers cargados: {len(data)}/{len(E2_TICKERS)}")

---
## 1. Datos Crudos — Vista Panorámica

### 1.1 Calidad de datos

In [ ]:
# --- Diagnóstico de calidad por ticker ---
# diagnose_data_quality() analiza: nulls por columna, días con volumen=0,
# timestamps duplicados, y gaps temporales >4 días.
# Nota: la función espera 'timestamp' como columna (no como índice),
# por eso hacemos reset_index() antes de pasarle el DataFrame.
quality_rows = []
for ticker, df in data.items():
    df_reset = df.reset_index()  # Mover timestamp de índice a columna
    report = diagnose_data_quality(df_reset, ticker)
    quality_rows.append({
        "Ticker": ticker,
        "Mercado": "AR" if ticker.endswith(".BA") else "US",
        "Filas": report["total_rows"],
        "Fecha inicio": df.index.min().strftime("%Y-%m-%d"),
        "Fecha fin": df.index.max().strftime("%Y-%m-%d"),
        "Nulls totales": sum(report["null_counts"].values()),  # Suma de nulls en todas las columnas
        "Vol=0 días": report["zero_volume_days"],   # Días sin negociación real
        "Gaps >4d": len(report["data_gaps_days"]),   # Huecos temporales sospechosos
        "Duplicados": report["duplicate_timestamps"], # Filas repetidas para la misma fecha
    })

# Mostrar como tabla con formato
quality_df = pd.DataFrame(quality_rows)
quality_df.style.set_caption("Resumen de calidad de datos — E2 Moderate")

### 1.2 Missing data heatmap

In [ ]:
# --- Heatmap de valores nulos ---
# Construimos una matriz donde cada fila es un ticker y cada columna es un campo OHLCV.
# El valor de cada celda es la cantidad de nulls. Idealmente todo debería ser 0
# después del proceso de limpieza (forward fill).
null_matrix = pd.DataFrame({
    ticker: df.isna().sum()   # Contar nulls por columna para cada ticker
    for ticker, df in data.items()
}).T  # Transponer: filas=tickers, columnas=campos OHLCV

fig, ax = plt.subplots(figsize=(10, 6))
# annot=True: muestra el número dentro de cada celda
# fmt=".0f": formato sin decimales
# cmap="YlOrRd": escala de color amarillo→naranja→rojo (más rojo = más nulls)
sns.heatmap(null_matrix, annot=True, fmt=".0f", cmap="YlOrRd", ax=ax,
            linewidths=0.5, cbar_kws={"label": "Cantidad de nulls"})
ax.set_title("Valores nulos por ticker y columna (post-cleaning)")
ax.set_ylabel("Ticker")
plt.tight_layout()
plt.show()

**Análisis — Calidad de datos**

Los datos post-limpieza deberían mostrar buena calidad general:

- **Cobertura temporal**: Los tickers US tienen ~10 años de datos diarios. Los AR pueden tener menor cobertura dependiendo de la fecha de inicio de cotización. Esto es suficiente para walk-forward con 5 folds.
- **Sin nulls ni duplicados** después del proceso de cleaning (forward fill). El heatmap debería confirmar 0 nulls en todas las columnas.
- **Gaps**: Algunos tickers AR pueden presentar más gaps >4 días debido a feriados argentinos adicionales y suspensiones de mercado. Esto es esperable y no afecta el modelado ya que trabajamos con datos diarios.

**Implicancia para el modelo**: Los tickers con menor cobertura temporal tendrán menos muestras de entrenamiento por fold en walk-forward, lo cual podría impactar su performance.

### 1.3 Series de precios normalizadas (base 100)

In [ ]:
# --- Series de precios normalizadas (base 100) ---
# Normalizar = dividir todos los precios por el precio del primer día y multiplicar por 100.
# Así todos arrancan en 100 y podemos comparar "cuánto creció cada acción" en el mismo gráfico,
# sin importar si una vale $5 y otra $500.
# Se separan en 2 paneles (AR vs US) porque las escalas son tan distintas
# que en un solo gráfico los US serían invisibles al lado de los AR.

fig, axes = plt.subplots(1, 2, figsize=(16, 6), sharey=False)  # sharey=False: cada panel tiene su propia escala Y

for ax, tickers, title in [
    (axes[0], AR_TICKERS, "Acciones Argentinas (.BA)"),
    (axes[1], US_TICKERS, "Acciones Estadounidenses"),
]:
    for ticker in tickers:
        if ticker not in data:
            continue
        close = data[ticker]["close"]
        # Normalización base 100: precio(t) / precio(t=0) * 100
        # Si sube a 200, significa que duplicó su valor desde el inicio
        normalized = 100 * close / close.iloc[0]
        ax.plot(normalized.index, normalized.values, label=ticker, linewidth=1.2)
    ax.set_title(title)
    ax.set_ylabel("Precio normalizado (base 100)")
    ax.legend(fontsize=9)
    ax.grid(True, alpha=0.3)

fig.suptitle("Evolución de precios — E2 Moderate", fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

**Análisis — Precios normalizados**

La evolución de precios revela dos universos muy distintos:

- **Mercado argentino**: Los tickers .BA (BBAR, BMA, EDN, TGSU2, LOMA) muestran retornos acumulados significativamente mayores en términos nominales, reflejando la **devaluación del peso argentino**. Dado que E2 se enfoca en tickers bancarios (BBAR, BMA) y de energía/infraestructura (EDN, TGSU2, LOMA), se esperan trayectorias volátiles con fuerte correlación con el riesgo-país.
- **Mercado estadounidense**: Los tickers US (NVDA, GOOGL, AMZN, META, NFLX) son empresas de tecnología/crecimiento (a diferencia de E1 que usa blue chips defensivos). Se espera mayor volatilidad que la canasta E1 (PG, JNJ) pero también mayor potencial de retorno.

**Implicancia para el modelo**: Las escalas de retorno son radicalmente distintas entre mercados. Esto justifica la normalización Z-score por ticker (no pooled) durante el entrenamiento, para que el modelo no esté dominado por la magnitud nominal de los tickers argentinos.

### 1.4 Distribución de retornos diarios

In [ ]:
# --- Histogramas de retornos diarios (log) con KDE y fit normal ---
# Para cada ticker, graficamos 3 capas superpuestas:
#   1. Histograma: barras que muestran la frecuencia de cada rango de retorno
#   2. KDE (Kernel Density Estimation): curva suavizada que estima la distribución real
#   3. Fit Normal: campana de Gauss con la misma media y std, para comparar

# Grilla de 2 filas x 5 columnas = 10 subplots (uno por ticker)
fig, axes = plt.subplots(2, 5, figsize=(20, 8))
axes = axes.flatten()  # Convertir matriz 2D de axes a lista 1D para iterar fácilmente

for i, ticker in enumerate(E2_TICKERS):
    if ticker not in data:
        continue
    ax = axes[i]

    # Retorno logarítmico diario = ln(precio_hoy) - ln(precio_ayer) = ln(precio_hoy / precio_ayer)
    # .diff() calcula la diferencia con el día anterior. .dropna() elimina el primer NaN.
    log_ret = np.log(data[ticker]["close"]).diff().dropna()

    # Capa 1: Histograma con 80 barras
    # density=True normaliza para que el área total sea 1 (probabilidad, no conteos)
    ax.hist(log_ret, bins=80, density=True, alpha=0.6, color="steelblue", edgecolor="none")

    # Capa 2: KDE (Kernel Density Estimation) - versión "suavizada" del histograma
    log_ret.plot.kde(ax=ax, color="darkblue", linewidth=1.5, label="KDE")

    # Capa 3: Curva normal teórica con la misma media (mu) y desviación estándar (sigma)
    # Si los datos fueran normales, la KDE azul coincidiría con esta curva roja
    mu, sigma = log_ret.mean(), log_ret.std()
    x = np.linspace(log_ret.min(), log_ret.max(), 200)  # 200 puntos para la curva
    ax.plot(x, stats.norm.pdf(x, mu, sigma), "r--", linewidth=1, label="Normal fit")

    # Estadísticos en el título:
    # sk (skewness): asimetría de retornos. 0=simétrico, <0=cola izquierda pesada, >0=cola derecha pesada
    # ku (excess kurtosis): peso de las colas. 0=normal, >0=colas más pesadas que la normal
    sk = stats.skew(log_ret)
    ku = stats.kurtosis(log_ret)
    ax.set_title(f"{ticker}\nsk={sk:.2f} ku={ku:.1f}", fontsize=10)
    ax.set_xlim(-0.15, 0.15)  # Limitar eje X para ver bien la zona central
    if i == 0:
        ax.legend(fontsize=7)

fig.suptitle("Distribución de retornos diarios (log) — E2 Moderate", fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# --- Q-Q Plots (Quantile-Quantile) vs distribución normal ---
# Un Q-Q plot compara los cuantiles de nuestros datos (eje Y) contra los cuantiles teóricos
# de una distribución normal (eje X). Si los datos fueran normales, los puntos caerían
# sobre la línea diagonal roja. Las desviaciones de la diagonal indican no-normalidad:
#   - Puntos arriba de la diagonal en las colas → colas más pesadas que la normal
#   - Puntos abajo de la diagonal en las colas → colas más ligeras que la normal

fig, axes = plt.subplots(2, 5, figsize=(20, 8))
axes = axes.flatten()

for i, ticker in enumerate(E2_TICKERS):
    if ticker not in data:
        continue
    ax = axes[i]
    log_ret = np.log(data[ticker]["close"]).diff().dropna()
    # probplot() calcula los cuantiles y dibuja el scatter + línea de referencia
    stats.probplot(log_ret, dist="norm", plot=ax)
    ax.set_title(f"{ticker}", fontsize=10)
    ax.get_lines()[0].set_markersize(2)  # Puntos más pequeños para mejor visualización

fig.suptitle("Q-Q Plots vs Normal — Retornos diarios E2", fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

**Análisis — Distribución de retornos diarios**

Los histogramas y Q-Q plots confirman un hallazgo clásico en finanzas: **los retornos NO siguen una distribución normal**.

**Evidencia cuantitativa:**
- **Excess kurtosis** elevada en todos los tickers: valores >3 indican colas pesadas (eventos extremos más frecuentes de lo que la normal predice).
- Los tickers AR (bancarios: BBAR, BMA; energía: EDN, TGSU2, LOMA) tienden a tener kurtosis más alta que los US, reflejando la mayor volatilidad del mercado argentino.
- Los tickers US de E2 (NVDA, GOOGL, AMZN, META, NFLX) son empresas de **alto crecimiento/tech**, por lo que se espera mayor kurtosis que las blue chips defensivas de E1 (JNJ, PG).
- **Test de Jarque-Bera**: p=0.0 para todos los tickers, rechazando normalidad con total confianza.
- Los **Q-Q plots** muestran desviación clara en ambas colas (puntos se alejan de la diagonal roja), confirmando las colas pesadas visualmente.

**Implicancia para el modelo**: 
1. La no-normalidad justifica usar **loss functions robustas** (Huber) en lugar de MSE puro, ya que MSE penaliza excesivamente los outliers.
2. La alta kurtosis justifica usar **modelos no lineales** (LSTM) que puedan capturar relaciones complejas en las colas.

### 1.5 Distribución de volumen (log-scale)

In [ ]:
# --- Boxplot de volumen en escala logarítmica ---
# El volumen (cantidad de acciones negociadas por día) varía enormemente entre mercados:
# una acción US puede negociar 100 millones de acciones/día, mientras una AR negocia 100 mil.
# Usamos log₁₀ para comprimir estas diferencias y poder compararlas visualmente.
# log₁₀(1,000) = 3, log₁₀(1,000,000) = 6, log₁₀(100,000,000) = 8

vol_data = []
for ticker, df in data.items():
    v = df["volume"].copy()
    v = v[v > 0]  # Filtrar días con volumen=0 para evitar log(0) = -infinito
    vol_data.append(pd.DataFrame({
        "log_volume": np.log10(v),  # Convertir a escala logarítmica base 10
        "Ticker": ticker,
        "Mercado": "AR" if ticker.endswith(".BA") else "US"
    }))

# Concatenar todos los tickers en un solo DataFrame para seaborn
vol_df = pd.concat(vol_data, ignore_index=True)

fig, ax = plt.subplots(figsize=(14, 6))
palette = {"AR": "#E74C3C", "US": "#3498DB"}  # Rojo para AR, Azul para US
# Boxplot: la caja muestra Q1-Q3 (50% central), la línea es la mediana,
# los bigotes van hasta 1.5×IQR, y los puntos fuera son outliers
sns.boxplot(data=vol_df, x="Ticker", y="log_volume", hue="Mercado",
            palette=palette, ax=ax, dodge=False)
ax.set_title("Distribución de volumen (log10) por ticker")
ax.set_ylabel("log10(Volumen)")
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

**Análisis — Volumen de negociación**

El boxplot en escala logarítmica revela diferencias estructurales de liquidez entre mercados:

- **Mercado US**: Los tickers tech/growth de E2 (NVDA, GOOGL, AMZN, META, NFLX) tienen volúmenes típicamente altos, con NVDA particularmente activa por el boom de IA.
- **Mercado AR**: Volúmenes típicamente **2-3 órdenes de magnitud menores** que los US. Los bancarios (BBAR, BMA) suelen tener mayor liquidez que las utilities (EDN, TGSU2, LOMA).
- La **dispersión** (largo del boxplot) puede ser mayor en tickers AR, indicando mayor variabilidad día a día en la actividad de trading.

**Implicancia para el modelo**: El feature `vol_ratio_20d` (ratio volumen actual / SMA de volumen 20d) normaliza estas diferencias de escala, permitiendo comparar anomalías de volumen independientemente del mercado. A diferencia de E1 que usa `vol_zscore_60`, E2 usa una ventana de 20 días más reactiva, acorde a su horizonte más corto.

---
## 2. Análisis de Features

### 2.1 Computar features y target

**Nota sobre el warm-up**: Se pierden ~220 muestras por ticker debido al período de calentamiento de los indicadores técnicos (el más largo es SMA_200, que necesita 200 días) más los 20 días del target forward. Esto reduce el dataset en ~220 filas. Con 5 folds de walk-forward, cada fold tiene suficientes muestras para evaluación estadística.

**Features detectados: 16** indicadores técnicos agrupados en 8 categorías:
- **Retornos** (4): ret_1d, ret_5d, ret_10d, ret_20d
- **Volatilidad** (2): vol_20d, atr_14
- **Volumen** (1): vol_ratio_20d
- **Tendencia** (3): sma50_sma200_ratio, close_sma50_dist, close_sma200_dist
- **Momentum** (2): macd_hist, rsi_14
- **Bollinger Bands** (2): bb_pct_b, bb_bandwidth
- **Fuerza de tendencia** (1): adx_14
- **Riesgo/asimetría** (1): skew_ret_20d

Comparado con E1 (15 features), E2 agrega `ret_1d` (retorno diario, informativo para H=20), `rsi_14` (sobrecompra/sobreventa) y `skew_ret_20d` (asimetría de distribución). No incluye `sma_50` ni `sma_200` como features directos (solo sus ratios y distancias).

In [ ]:
# --- Computar features técnicos y target para todos los tickers ---
# compute_e2_features(df) calcula 16 indicadores técnicos a partir de OHLCV:
#   - Retornos (ret_1d, ret_5d, ret_10d, ret_20d), Volatilidad (vol_20d, atr_14),
#   - Volumen (vol_ratio_20d), Tendencia (sma50_sma200_ratio, close_sma50_dist, close_sma200_dist),
#   - Momentum (macd_hist, rsi_14), Bollinger (bb_pct_b, bb_bandwidth),
#   - Fuerza de tendencia (adx_14), Riesgo (skew_ret_20d)
#
# make_target_e2(df, 20) calcula el retorno logarítmico a 20 días hacia el futuro:
#   target_ret_20d = ln(precio_en_20_días / precio_hoy)
#
# Importante: Al combinar features + target, se pierden ~220 filas por ticker:
#   - ~200 filas al inicio: "warm-up" de SMA_200 (necesita 200 días de historia)
#   - ~20 filas al final: no hay datos futuros para calcular el target

features_all = {}  # {ticker: DataFrame con features}
targets_all = {}   # {ticker: Series con target}

for ticker, df in data.items():
    feat = compute_e2_features(df)   # DataFrame con 16 columnas de features
    tgt = make_target_e2(df, HORIZON)  # Series con retorno a 20 días

    # Combinar features y target en un solo DataFrame y eliminar filas con NaN
    # (las primeras ~200 por warm-up de indicadores, las últimas ~20 por el target forward)
    combined = pd.concat([feat, tgt], axis=1).dropna()

    features_all[ticker] = combined[feat.columns]  # Solo las columnas de features
    targets_all[ticker] = combined[tgt.name]        # Solo la columna del target

    print(f"  {ticker}: {len(combined):,} muestras válidas (de {len(df):,} originales, "
          f"warm-up ~{len(df)-len(combined)} descartadas)")

# Guardar nombres de features para usar en gráficos posteriores
FEATURE_NAMES = list(features_all[E2_TICKERS[0]].columns)
print(f"\nFeatures ({len(FEATURE_NAMES)}): {FEATURE_NAMES}")

### 2.2 Matriz de correlación entre features

In [ ]:
# --- Matriz de correlación entre features + target ---
# "Poolear" = juntar los datos de todos los tickers en un solo DataFrame grande.
# Esto da una visión global de cómo se relacionan los features entre sí y con el target.
all_feat_target = pd.concat([
    pd.concat([features_all[t], targets_all[t]], axis=1)
    for t in data.keys()
], ignore_index=True)  # ignore_index para reindexar de 0 a N

# Calcular la matriz de correlación de Pearson (r entre -1 y +1)
# r cercano a +1: cuando uno sube, el otro también (correlación positiva)
# r cercano a -1: cuando uno sube, el otro baja (correlación negativa)
# r cercano a 0: no hay relación lineal
corr = all_feat_target.corr()

fig, ax = plt.subplots(figsize=(16, 14))
# mask: solo mostrar el triángulo inferior (el superior es espejo)
mask = np.triu(np.ones_like(corr, dtype=bool), k=1)
# cmap="RdBu_r": rojo=correlación positiva, azul=negativa, blanco=0
sns.heatmap(corr, mask=mask, annot=True, fmt=".2f", cmap="RdBu_r",
            center=0, vmin=-1, vmax=1, ax=ax, linewidths=0.5,
            square=True, cbar_kws={"shrink": 0.8})
ax.set_title("Matriz de correlación — Features E2 + Target (datos pooled)", fontsize=13)
plt.tight_layout()
plt.show()

# --- Detectar pares con alta correlación (posible multicolinealidad) ---
# Multicolinealidad = dos features que miden casi lo mismo.
# Esto puede hacer que el modelo sea inestable (pesos oscilan entre ambos features).
# Umbral: |r| > 0.7 se considera "alta correlación"
high_corr = []
for i in range(len(corr.columns)):
    for j in range(i+1, len(corr.columns)):  # Solo triángulo superior para no repetir
        r = corr.iloc[i, j]
        if abs(r) > 0.7:
            high_corr.append((corr.columns[i], corr.columns[j], round(r, 3)))

if high_corr:
    print("\n⚠️ Pares con correlación |r| > 0.7:")
    for f1, f2, r in sorted(high_corr, key=lambda x: abs(x[2]), reverse=True):
        print(f"  {f1} ↔ {f2}: r = {r}")
else:
    print("\n✓ No hay pares con correlación |r| > 0.7")

**Análisis — Matriz de correlación**

Observaciones esperadas sobre la estructura de correlación de los 16 features de E2:

**Multicolinealidades a monitorear:**
- `ret_5d` / `ret_10d` / `ret_20d` comparten ventanas solapadas con `ret_1d`, por lo que se esperan correlaciones moderadas-altas entre los features de retorno.
- `vol_20d` y `atr_14`: ambos miden volatilidad (uno basado en retornos, otro en true range), probablemente correlacionados.
- `vol_20d` y `bb_bandwidth`: ambos dependen de la volatilidad de 20 días.
- `close_sma50_dist` y `close_sma200_dist`: ambos miden distancia del precio a promedios móviles.

**Features con mayor correlación con el target** (`target_ret_20d`):
- Se espera que los features de volatilidad (`atr_14`, `vol_20d`) y tendencia (`sma50_sma200_ratio`) tengan las correlaciones más altas con el target, aunque probablemente menores que en E1 dada la mayor dificultad de predecir a 20 días (menor signal-to-noise ratio a corto plazo).
- `rsi_14` podría mostrar una señal contrarian (sobrecompra → menor retorno futuro), mientras que los features de momentum podrían mostrar señal de continuación.

**Nota**: A diferencia de E1, E2 no incluye `sma_50` y `sma_200` como features directos (solo sus ratios y distancias), lo cual debería reducir la multicolinealidad severa observada en E1 (r=0.987 entre sma_50 y sma_200).

### 2.3 Feature vs Target (scatter + regresión)

In [ ]:
# --- Scatter plots: cada feature vs el target ---
# Cada subplot muestra un feature en eje X y el target (retorno a 20d) en eje Y.
# Si hay una relación lineal, los puntos seguirán una tendencia y r será alto.
# En finanzas, r > 0.1 ya se considera una señal interesante.

# Calcular cuántas filas y columnas necesitamos para la grilla
n_features = len(FEATURE_NAMES)
ncols = 4
nrows = (n_features + ncols - 1) // ncols  # Redondeo hacia arriba

fig, axes = plt.subplots(nrows, ncols, figsize=(20, 5 * nrows))
axes = axes.flatten()

target_name = f"target_ret_{HORIZON}d"

for i, feat_name in enumerate(FEATURE_NAMES):
    ax = axes[i]
    x = all_feat_target[feat_name]  # Valores del feature (pooled, todos los tickers)
    y = all_feat_target[target_name]  # Valores del target

    # Subsampling: si hay más de 3000 puntos, tomar una muestra aleatoria
    # para que el gráfico no sea una mancha ilegible
    if len(x) > 3000:
        idx = np.random.RandomState(42).choice(len(x), 3000, replace=False)
        x_plot, y_plot = x.iloc[idx], y.iloc[idx]
    else:
        x_plot, y_plot = x, y

    # Scatter plot: cada punto es un día de un ticker
    ax.scatter(x_plot, y_plot, alpha=0.15, s=8, color="steelblue")

    # Línea de regresión lineal (y = mx + b) ajustada a TODOS los datos (no solo el subsample)
    m, b = np.polyfit(x.values, y.values, 1)
    x_line = np.linspace(x.quantile(0.01), x.quantile(0.99), 100)  # Rango sin outliers extremos
    ax.plot(x_line, m * x_line + b, "r-", linewidth=1.5)

    # Correlación de Pearson (r) y su p-value
    # r: fuerza y dirección de la relación lineal (-1 a +1)
    # p: probabilidad de obtener este r por azar (p < 0.05 = estadísticamente significativo)
    r, p = stats.pearsonr(x, y)
    ax.set_title(f"{feat_name}\nr = {r:.3f} (p = {p:.1e})", fontsize=10)
    ax.set_xlabel(feat_name, fontsize=8)
    if i % ncols == 0:
        ax.set_ylabel(target_name, fontsize=8)

# Ocultar subplots vacíos si n_features no es múltiplo de ncols
for j in range(n_features, len(axes)):
    axes[j].set_visible(False)

fig.suptitle(f"Feature vs {target_name} — Pearson correlation (datos pooled)", fontsize=14, y=1.01)
plt.tight_layout()
plt.show()

**Análisis — Feature vs Target**

Los scatter plots muestran las relaciones lineales entre cada feature individual y el target a 20 días:

- **Correlaciones esperadas más fuertes**: Volatilidad (`atr_14`, `vol_20d`, `bb_bandwidth`) y tendencia (`sma50_sma200_ratio`) deberían mostrar las correlaciones más significativas.
- **RSI_14**: Se espera una relación no lineal (posiblemente contrarian: RSI alto → menor retorno futuro), lo cual no se captura bien en la correlación de Pearson lineal. El modelo LSTM puede detectar este patrón.
- **Nubes dispersas**: Ningún feature por sí solo debería explicar más del ~5-10% de la varianza del target. Esto es normal para H=20 días (más ruido que H=90).

**Implicancia para el modelo**: La baja predictibilidad individual de cada feature confirma que el valor del modelo LSTM está en capturar **interacciones no lineales y patrones temporales** entre features, no en relaciones lineales simples. Para un horizonte de 20 días, se espera aún mayor dificultad que para 90 días.

---
## 3. Análisis del Target

### 3.1 Distribución del target por ticker

In [ ]:
# --- Distribución del target (retorno a 20 días) por ticker ---
# Misma estructura que los histogramas de retornos diarios (histograma + KDE + normal fit),
# pero ahora para el target: retorno logarítmico acumulado en 20 días hacia el futuro.
# Ejemplo: si target = 0.05, significa que el precio subió ~5.1% en los próximos 20 días
#          (e^0.05 ≈ 1.051, o sea +5.1%)

fig, axes = plt.subplots(2, 5, figsize=(20, 8))
axes = axes.flatten()

target_name = f"target_ret_{HORIZON}d"

for i, ticker in enumerate(E2_TICKERS):
    if ticker not in targets_all:
        continue
    ax = axes[i]
    tgt = targets_all[ticker]

    # Histograma del target
    ax.hist(tgt, bins=60, density=True, alpha=0.6, color="steelblue", edgecolor="none")
    # KDE suavizada
    tgt.plot.kde(ax=ax, color="darkblue", linewidth=1.5)

    # Curva normal teórica para comparar
    mu, sigma = tgt.mean(), tgt.std()
    x = np.linspace(tgt.min(), tgt.max(), 200)
    ax.plot(x, stats.norm.pdf(x, mu, sigma), "r--", linewidth=1, label="Normal")

    # Estadísticos en el título:
    # μ (mu): retorno medio a 20 días. Ej: 0.05 = +5% promedio
    # σ (sigma): dispersión. Cuanto mayor, más difícil de predecir
    sk = stats.skew(tgt)
    ku = stats.kurtosis(tgt)
    ax.set_title(f"{ticker}\nμ={mu:.3f} σ={sigma:.3f}\nsk={sk:.2f} ku={ku:.1f}", fontsize=9)
    if i == 0:
        ax.legend(fontsize=7)

fig.suptitle(f"Distribución de {target_name} por ticker", fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

**Análisis — Distribución del target (retorno a 20 días)**

El target a 20 días muestra características distintas al target de 90 días de E1:

**Comparación con E1 (90 días):**
- **Menor media**: Al ser un horizonte más corto, los retornos acumulados son proporcionalmente menores (tanto la señal como el drift inflacionario).
- **Menor dispersión (σ)**: Con 20 días vs 90 días, la varianza del target es menor, lo cual reduce la incertidumbre pero también la señal.
- **Distribuciones más normales**: El efecto de agregación temporal es menor con 20 días, pero aún debería generar distribuciones más cercanas a la normal que los retornos diarios.

**Mercado AR vs US:**
- Los tickers AR (BBAR, BMA, EDN, TGSU2, LOMA) deberían mostrar media positiva por el efecto inflación/devaluación, pero menos extrema que a 90 días.
- Los tickers US tech (NVDA, GOOGL, AMZN, META, NFLX) podrían mostrar mayor dispersión que las blue chips de E1, dado su perfil de crecimiento.

**Implicancia para el modelo**: La normalización Z-score por ticker sigue siendo esencial dada la diferencia de escala entre mercados.

### 3.2 Target en el tiempo

In [ ]:
# --- Target en el tiempo: regímenes positivos y negativos ---
# Para cada ticker, graficamos el target_ret_20d como serie temporal.
# Verde = períodos donde el retorno a 20d fue positivo (mercado alcista)
# Rojo = períodos donde el retorno a 20d fue negativo (mercado bajista)
# Esto permite ver visualmente los "regímenes" de mercado.

fig, axes = plt.subplots(5, 2, figsize=(18, 20), sharex=True)  # sharex: mismo eje X (tiempo)
axes = axes.flatten()

for i, ticker in enumerate(E2_TICKERS):
    if ticker not in targets_all:
        continue
    ax = axes[i]
    tgt = targets_all[ticker]

    # fill_between: colorea el área entre la línea y el eje Y=0
    # where=tgt >= 0: solo colorear donde el target es positivo (verde)
    ax.fill_between(tgt.index, tgt.values, 0,
                    where=tgt.values >= 0, color="green", alpha=0.3, label="Positivo")
    # where=tgt < 0: solo colorear donde el target es negativo (rojo)
    ax.fill_between(tgt.index, tgt.values, 0,
                    where=tgt.values < 0, color="red", alpha=0.3, label="Negativo")
    # Línea delgada negra para ver el contorno exacto
    ax.plot(tgt.index, tgt.values, color="black", linewidth=0.5, alpha=0.5)
    ax.axhline(0, color="gray", linewidth=0.5, linestyle="--")  # Línea horizontal en Y=0
    ax.set_title(f"{ticker}", fontsize=11)
    ax.set_ylabel(f"ret_{HORIZON}d")
    if i == 0:
        ax.legend(fontsize=8)

fig.suptitle(f"{target_name} en el tiempo — Regímenes positivos y negativos", fontsize=14, y=1.01)
plt.tight_layout()
plt.show()

**Análisis — Target en el tiempo**

La serie temporal del target a 20 días revela **regímenes de mercado**, aunque con transiciones más rápidas que el target de 90 días de E1:

- **Alternancia más frecuente**: Con H=20 días, los cambios de signo (positivo ↔ negativo) ocurren con mayor frecuencia que con H=90. Esto refleja la naturaleza más volátil del horizonte corto.
- **Caídas sincronizadas**: Los eventos sistémicos (COVID-2020, correcciones de mercado) deberían ser visibles como períodos rojos simultáneos en todos los tickers.
- **Tickers AR vs US**: Los AR muestran mayor amplitud de oscilación y los tickers bancarios (BBAR, BMA) pueden mostrar correlación con el riesgo soberano argentino.
- **Tickers US tech**: NVDA, META y NFLX pueden mostrar mayor volatilidad en el target que las blue chips de E1, dado su perfil de crecimiento.

**Implicancia para el modelo**: La presencia de regímenes justifica el **walk-forward validation** y el **rebalanceo semanal** (vs mensual de E1) para adaptarse más rápido a cambios de régimen.

### 3.3 Autocorrelación del target (ACF / PACF)

In [ ]:
# --- ACF / PACF del target ---
# ACF (Autocorrelation Function): mide la correlación del target consigo mismo
# desplazado en el tiempo (lags). Si ACF(lag=5) = 0.9, significa que el target
# de hoy y el de hace 5 días están 90% correlacionados.
#
# PACF (Partial ACF): similar pero "controlando" por los lags intermedios.
# Si PACF(lag=5) es alto pero PACF(lag=2,3,4) son bajos, la correlación en lag 5
# no se explica solo por la transmisión a través de lags 1→2→3→4→5.
#
# Se muestran 4 tickers representativos (2 AR + 2 US) para no sobrecargar el gráfico.

sample_tickers = [AR_TICKERS[0], AR_TICKERS[1], US_TICKERS[0], US_TICKERS[1]]
sample_tickers = [t for t in sample_tickers if t in targets_all]

fig, axes = plt.subplots(len(sample_tickers), 2, figsize=(16, 4 * len(sample_tickers)))

for i, ticker in enumerate(sample_tickers):
    tgt = targets_all[ticker].dropna()

    # lags=40: analizar correlación hasta 40 días de desfase
    # La banda azul sombreada indica el intervalo de confianza al 95%:
    # barras fuera de la banda = correlación estadísticamente significativa
    plot_acf(tgt, ax=axes[i, 0], lags=40, title=f"{ticker} — ACF")
    plot_pacf(tgt, ax=axes[i, 1], lags=40, title=f"{ticker} — PACF", method="ywm")

fig.suptitle(f"Autocorrelación de {target_name} (40 lags)", fontsize=14, y=1.01)
plt.tight_layout()
plt.show()

print("Nota: Alta autocorrelación es esperada porque el target usa ventanas de 20 días\n"
      "superpuestas (retornos calculados diariamente). Dos observaciones consecutivas\n"
      "comparten 19 de 20 días → correlación mecánica alta (pero menor que E1, donde\n"
      "se comparten 89 de 90 días). El embargo de 20 días en walk-forward lo maneja.")

**Análisis — Autocorrelación del target**

Los gráficos ACF y PACF muestran **alta autocorrelación** en el target, con un patrón similar pero menos extremo que E1:

- **ACF**: Los coeficientes se mantienen significativos hasta el lag ~20 y luego decaen. El decaimiento es más rápido que en E1 (donde persistía hasta lag 90+).
- **PACF**: Pico dominante en lag 1 con decaimiento rápido, indicando que la autocorrelación se transmite a través de observaciones consecutivas.

**Causa**: Autocorrelación **mecánica, no informativa**. El target `target_ret_20d` se calcula como retorno acumulado en una ventana deslizante de 20 días. Dos observaciones consecutivas comparten 19 de 20 días, generando correlación mecánica alta (pero proporcionalmente menor que E1 donde se comparten 89/90 días).

**Consecuencia**: El **embargo de 20 días** configurado en `base.yaml` (`splits.embargo_days.e2 = 20`) elimina esta dependencia mecánica, garantizando que ninguna observación de test comparta días con el train set.

**Esto NO es leakage del modelo**, sino una propiedad intrínseca de targets con ventanas superpuestas.

### 3.4 Rolling mean y std del target

In [ ]:
# --- Rolling mean y std del target ---
# "Rolling" = calcular un estadístico en una ventana deslizante de N días.
# Esto muestra cómo cambia la media y la volatilidad del target a lo largo del tiempo.
# Si la media y std fueran constantes, la serie sería "estacionaria".
# En la práctica, cambian mucho → el target es NO estacionario.

ROLLING_W = 60  # Ventana de 60 días (~3 meses)

fig, axes = plt.subplots(5, 2, figsize=(18, 20), sharex=True)
axes = axes.flatten()

for i, ticker in enumerate(E2_TICKERS):
    if ticker not in targets_all:
        continue
    ax = axes[i]
    tgt = targets_all[ticker]

    # Media móvil: promedio del target en los últimos 60 días
    # Si sube → estamos en un período alcista. Si baja → bajista.
    roll_mean = tgt.rolling(ROLLING_W).mean()
    # Desviación estándar móvil: volatilidad del target en los últimos 60 días
    # Banda ancha = alta incertidumbre. Banda estrecha = mercado estable.
    roll_std = tgt.rolling(ROLLING_W).std()

    # Línea azul: media móvil
    ax.plot(roll_mean.index, roll_mean.values, color="blue", linewidth=1.2,
            label=f"Media móvil ({ROLLING_W}d)")
    # Banda azul sombreada: media ± 1 desviación estándar
    # Cubre ~68% de las observaciones si la distribución fuera normal
    ax.fill_between(roll_mean.index,
                    (roll_mean - roll_std).values,
                    (roll_mean + roll_std).values,
                    alpha=0.2, color="blue", label="±1 std")
    ax.axhline(0, color="gray", linewidth=0.5, linestyle="--")
    ax.set_title(f"{ticker}", fontsize=11)
    ax.set_ylabel(f"ret_{HORIZON}d")
    if i == 0:
        ax.legend(fontsize=8)

fig.suptitle(f"Rolling mean ± std de {target_name} (ventana={ROLLING_W}d)\n"
             f"No-estacionaridad y clusters de volatilidad",
             fontsize=14, y=1.01)
plt.tight_layout()
plt.show()

**Análisis — Rolling mean y std del target**

Los gráficos de media móvil y banda de ±1 std confirman la **no-estacionaridad** del target:

- **Media variable en el tiempo**: La media del retorno a 20 días oscila entre valores positivos y negativos, con transiciones más frecuentes que el target de 90 días de E1.
- **Volatilidad variable** (clusters): Los períodos de alta volatilidad (bandas anchas) se alternan con períodos de baja volatilidad (bandas estrechas). Esto se conoce como **volatility clustering**.
- **Diferencia AR vs US**: Los tickers AR muestran bandas más anchas y mayor variabilidad. Los tickers US tech pueden mostrar clusters de volatilidad asociados a reportes de earnings.
- **COVID-2020**: Visible como un pico de volatilidad en todos los tickers.

**Implicancia para el modelo**: 
1. La no-estacionaridad confirma que un modelo entrenado en un régimen puede fallar en otro. El **walk-forward** con reentrenamiento periódico mitiga esto.
2. El rebalanceo **semanal** (vs mensual de E1) permite adaptarse más rápido a cambios de régimen del target a 20 días.

---
## 4. Resumen estadístico

### 4.1 Tabla de estadísticos por ticker

In [ ]:
# --- Tabla resumen de estadísticos por ticker ---
# Calcula métricas descriptivas para retornos diarios y para el target (20d).
# Esto permite comparar cuantitativamente las diferencias entre tickers y mercados.

summary_rows = []
for ticker in E2_TICKERS:
    if ticker not in data:
        continue
    df = data[ticker]
    log_ret = np.log(df["close"]).diff().dropna()  # Retornos diarios
    tgt = targets_all.get(ticker)

    # Test de Jarque-Bera: contrasta la hipótesis "los datos son normales"
    # Si p < 0.05 → rechazamos normalidad (los datos NO son normales)
    # En finanzas, prácticamente siempre p ≈ 0 (nunca son normales)
    jb_stat, jb_p = stats.jarque_bera(log_ret)

    row = {
        "Ticker": ticker,
        "Mercado": "AR" if ticker.endswith(".BA") else "US",
        "N obs": len(df),
        # --- Estadísticos de retornos diarios ---
        "Ret diario μ": f"{log_ret.mean():.5f}",      # Retorno promedio diario
        "Ret diario σ": f"{log_ret.std():.4f}",        # Volatilidad diaria
        "Skewness": f"{stats.skew(log_ret):.2f}",      # Asimetría (<0 = caídas más severas)
        "Kurtosis (exc)": f"{stats.kurtosis(log_ret):.1f}",  # Colas pesadas (>0 = más extremos)
        "JB p-value": f"{jb_p:.1e}",                    # Test normalidad (p<0.05 → no normal)
    }

    if tgt is not None:
        row.update({
            # --- Estadísticos del target (retorno a 20 días) ---
            "Target μ": f"{tgt.mean():.4f}",    # Retorno promedio a 20d
            "Target σ": f"{tgt.std():.4f}",      # Volatilidad del target
            "Target min": f"{tgt.min():.3f}",    # Peor retorno a 20d observado
            "Target max": f"{tgt.max():.3f}",    # Mejor retorno a 20d observado
        })

    summary_rows.append(row)

summary_df = pd.DataFrame(summary_rows)
summary_df.style.set_caption("Estadísticos por ticker — E2 Moderate")

**Análisis — Tabla de estadísticos**

La tabla confirma cuantitativamente las observaciones previas:

- **Retorno diario medio**: AR > US. Los tickers argentinos tienen un drift positivo mayor, dominado por la inflación/devaluación.
- **Volatilidad diaria**: AR > US. Los tickers bancarios AR (BBAR, BMA) y los US tech (NVDA) suelen ser más volátiles.
- **No-normalidad universal**: Jarque-Bera p≈0 para todos los tickers. Ninguno pasa el test de normalidad.
- **Target a 20 días**: La dispersión (σ) del target debería ser menor que la de E1 (H=90) proporcionalmente, pero la relación señal/ruido también es menor.

### 4.2 Heatmap cross-ticker de features

In [ ]:
# --- Heatmap cross-ticker: media de cada feature por ticker ---
# Cada celda muestra la media de un feature para un ticker específico.
# Como los features tienen escalas muy distintas (ej: rsi_14 ~ 50 vs bb_pct_b ~ 0-1),
# estandarizamos por columna (z-score entre tickers) para poder comparar visualmente.
# Z-score = (valor - media_entre_tickers) / std_entre_tickers
# Valores extremos (|z| > 2) indican tickers que se comportan muy diferente del grupo.

# Calcular la media de cada feature para cada ticker
feat_means = pd.DataFrame({
    ticker: features_all[ticker].mean()
    for ticker in E2_TICKERS if ticker in features_all
}).T  # Transponer: filas=tickers, columnas=features

# Estandarizar por columna (z-score entre tickers, no entre features)
# Así podemos ver qué tickers son atípicos para cada feature
feat_means_z = (feat_means - feat_means.mean()) / (feat_means.std() + 1e-12)

fig, ax = plt.subplots(figsize=(16, 7))
# cmap="RdBu_r": rojo = z-score positivo (mayor que el promedio del grupo)
#                azul = z-score negativo (menor que el promedio del grupo)
sns.heatmap(feat_means_z, annot=True, fmt=".2f", cmap="RdBu_r", center=0,
            ax=ax, linewidths=0.5, cbar_kws={"label": "Z-score (relativo entre tickers)"})
ax.set_title("Media de features por ticker (z-score entre tickers)\n"
             "Valores extremos indican tickers atípicos", fontsize=13)
ax.set_ylabel("Ticker")
plt.tight_layout()
plt.show()

---
## 5. Conclusiones del EDA

### Hallazgos principales

1. **Los retornos financieros no son normales**: Todos los tickers rechazan normalidad (Jarque-Bera p≈0) con excess kurtosis significativa. Esto justifica el uso de **loss function Huber** (robusta a outliers) y **modelos no lineales** (LSTM) en lugar de regresión lineal.

2. **Dos universos con escalas distintas**: Los tickers argentinos tienen retornos nominales mayores y volatilidad mayor que los estadounidenses, debido al efecto inflación/devaluación del peso. Esto valida la decisión de:
   - Normalizar features y target con **Z-score por ticker** (no pooled)
   - Entrenar **modelos independientes** por ticker

3. **Universo de tickers diferenciado vs E1**: E2 usa tickers bancarios/energía AR (BBAR, BMA, EDN, TGSU2, LOMA) y tech/growth US (NVDA, GOOGL, AMZN, META, NFLX), a diferencia de E1 que usa blue chips defensivos. Esto implica mayor volatilidad general y mayor potencial de retorno, acorde al perfil "moderado" de la estrategia.

4. **Diseño de features orientado al horizonte**: Los 16 features de E2 están diseñados para H=20 días, con ventanas más cortas (5d, 10d, 20d) que E1 (1w, 4w, 13w). La inclusión de `rsi_14` (sobrecompra/sobreventa) y `skew_ret_20d` (asimetría) son diferenciadores clave respecto a E1, capturando señales relevantes para el horizonte más corto.

5. **No-estacionaridad y regímenes**: La media y varianza del target a 20 días cambian en el tiempo, con transiciones más frecuentes que H=90. Esto justifica el **walk-forward validation** con **rebalanceo semanal** (vs mensual de E1).

6. **Autocorrelación mecánica del target**: Esperada por las ventanas superpuestas de 20 días (19/20 días compartidos entre observaciones consecutivas). El embargo de 20 días configurado en el pipeline lo maneja correctamente.

### Recomendaciones para el modelado

- **Monitorear multicolinealidad**: Los features de retorno (ret_1d, ret_5d, ret_10d, ret_20d) tienen ventanas solapadas que generan correlación mecánica. Evaluar si reducir a 2-3 features de retorno mejora la estabilidad.
- **RSI como señal contrarian**: El RSI_14 podría capturar reversiones de medio plazo (sobrecompra → caída, sobreventa → rebote) que complementen las señales de momentum. El LSTM puede modelar esta no-linealidad.
- **Volatilidad de tickers US tech**: NVDA y NFLX pueden mostrar volatilidad significativamente mayor que los otros US tickers, lo cual podría impactar las métricas agregadas. Considerar monitorear métricas por ticker además del promedio.